# Chest X-Ray Multi-Label Classification (MIMIC-CXR-JPG)

DenseNet121 encoder, 13 labels (12 findings + No Finding).
This classifier becomes the diagnosis-aware front end of the report generation model in the next stage.

Run order: Setup -> EDA -> Data -> Model -> Train -> Results -> Test

## 0. Create the helper module

Run this once. It writes `cxr_data.py` next to the notebook so that the DataLoader
workers can import the Dataset class (required on Windows).

In [ ]:
%%writefile cxr_data.py
import os

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

LABEL_COLS = [
    "No Finding",
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion",
    "Lung Opacity", "Pleural Effusion", "Pleural Other",
    "Pneumonia", "Pneumothorax",
]

IMG_SIZE = 224
_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    _NORM,
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    _NORM,
])


class CXRDataset(Dataset):
    def __init__(self, df, tf):
        self.paths = df["local_path"].tolist()
        self.labels = df[LABEL_COLS].values.astype("float32")
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        for k in range(10):
            j = (i + k) % len(self.paths)
            try:
                img = Image.open(self.paths[j]).convert("RGB")
            except Exception:
                continue
            return self.tf(img), torch.from_numpy(self.labels[j])
        raise RuntimeError("too many unreadable images near index %d" % i)


def load_frames(data_dir=".", image_root="images", labels_csv="available_labels.csv",
                val_fraction=0.05, seed=42):
    df = pd.read_csv(os.path.join(data_dir, labels_csv))
    if "local_path" not in df.columns:
        df["local_path"] = df["image_path"].apply(
            lambda p: os.path.join(data_dir, image_root, p)
        )
    for c in LABEL_COLS:
        df[c] = df[c].fillna(0).replace(-1, 0).astype("float32")

    train = df[df["split"] == "train"].reset_index(drop=True)
    val = df[df["split"] == "validate"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)

    if len(val) < 500:
        extra = train.sample(n=int(len(train) * val_fraction), random_state=seed)
        train = train.drop(extra.index).reset_index(drop=True)
        val = pd.concat([val, extra], ignore_index=True)

    return train, val, test

## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.metrics import roc_auc_score, roc_curve

from cxr_data import LABEL_COLS, CXRDataset, train_tf, eval_tf, load_frames

DATA_DIR    = "."
CKPT_DIR    = "checkpoints"
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 1e-4
NUM_WORKERS = 4
SEED        = 42

os.makedirs(CKPT_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load labels and explore

In [ ]:
train_df, val_df, test_df = load_frames(DATA_DIR)
print(f"train {len(train_df)}   val {len(val_df)}   test {len(test_df)}")
train_df.head(3)[["dicom_id", "split"] + LABEL_COLS[:4]]

In [ ]:
counts = train_df[LABEL_COLS].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(len(counts)), counts.values, color="#3A6EA5")
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=45, ha="right")
ax.set_ylabel("Positive images (train)")
ax.set_title("Class distribution")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

print("Imbalance ratio (max/min): %.1fx" % (counts.max() / max(counts.min(), 1)))

In [ ]:
n_labels = train_df[LABEL_COLS[1:]].sum(axis=1)
fig, ax = plt.subplots(figsize=(6, 3.5))
n_labels.value_counts().sort_index().plot(kind="bar", ax=ax, color="#8A938F")
ax.set_xlabel("Number of findings per image"); ax.set_ylabel("Images")
ax.set_title("Multi-label density")
plt.tight_layout(); plt.show()

In [ ]:
M = train_df[LABEL_COLS].astype(int).T.dot(train_df[LABEL_COLS].astype(int))
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(np.log1p(M.values), cmap="YlOrRd")
ax.set_xticks(range(len(LABEL_COLS))); ax.set_xticklabels(LABEL_COLS, rotation=90, fontsize=8)
ax.set_yticks(range(len(LABEL_COLS))); ax.set_yticklabels(LABEL_COLS, fontsize=8)
ax.set_title("Label co-occurrence (log scale)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
from PIL import Image

sample = train_df.sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(11, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open(row["local_path"]), cmap="gray")
    pos = [c for c in LABEL_COLS if row[c] == 1]
    ax.set_title(", ".join(pos) if pos else "(no positive label)", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Dataloaders

In [ ]:
train_loader = DataLoader(CXRDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(CXRDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0)

xb, yb = next(iter(train_loader))
print("batch images:", tuple(xb.shape), "| labels:", tuple(yb.shape))

## 4. Model and weighted loss

`pos_weight = negatives / positives` per class, capped at 20 so that rare classes
(Pleural Other, Fracture) contribute to the gradient without destabilising training.
This keeps the multi-label structure intact instead of resampling the dataset.

In [ ]:
model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
model.classifier = nn.Linear(model.classifier.in_features, len(LABEL_COLS))
model = model.to(device)

pos = train_df[LABEL_COLS].sum().values.astype("float32")
neg = len(train_df) - pos
pos_weight = torch.tensor(np.clip(neg / np.clip(pos, 1, None), 1.0, 20.0),
                          dtype=torch.float32, device=device)

for c, w in zip(LABEL_COLS, pos_weight.cpu().numpy()):
    print(f"  {c:30s} {w:6.2f}")

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler    = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

## 5. Train and validate

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(x)
        P.append(torch.sigmoid(logits.float()).cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    aucs = {c: roc_auc_score(Y[:, i], P[:, i])
            for i, c in enumerate(LABEL_COLS) if Y[:, i].min() != Y[:, i].max()}
    return float(np.mean(list(aucs.values()))), aucs, P, Y

In [ ]:
history   = []
best_auc  = 0.0
start_ep  = 1
last_path = os.path.join(CKPT_DIR, "last.pt")

if os.path.isfile(last_path):
    ck = torch.load(last_path, map_location=device)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_ep, best_auc, history = ck["epoch"] + 1, ck["best_auc"], ck.get("history", [])
    print(f"Resumed from epoch {ck['epoch']}, best mean AUC {best_auc:.4f}")

In [ ]:
for epoch in range(start_ep, EPOCHS + 1):
    model.train()
    running, seen, t0 = 0.0, 0, time.time()

    for step, (x, y) in enumerate(train_loader, 1):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()

        running += loss.item() * x.size(0); seen += x.size(0)
        if step % 50 == 0:
            print(f"  ep{epoch} {step}/{len(train_loader)}  loss {running/seen:.4f}"
                  f"  {seen/(time.time()-t0):.1f} img/s")

    train_loss = running / seen
    mean_auc, aucs, _, _ = evaluate(model, val_loader)
    scheduler.step(mean_auc)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_auc": mean_auc,
                    "per_class": aucs, "minutes": (time.time() - t0) / 60})
    print(f"epoch {epoch}: loss {train_loss:.4f}  val mean AUC {mean_auc:.4f}"
          f"  ({(time.time()-t0)/60:.1f} min)")

    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "best_auc": max(best_auc, mean_auc),
                "history": history, "labels": LABEL_COLS}, last_path)

    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save({"model": model.state_dict(), "labels": LABEL_COLS, "auc": best_auc},
                   os.path.join(CKPT_DIR, "best.pt"))
        print(f"    new best saved ({best_auc:.4f})")

json.dump(history, open(os.path.join(CKPT_DIR, "history.json"), "w"), indent=2)

## 6. Results

In [ ]:
history = json.load(open(os.path.join(CKPT_DIR, "history.json")))
ep   = [h["epoch"] for h in history]
loss = [h["train_loss"] for h in history]
auc  = [h["val_auc"] for h in history]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, loss, marker="o", color="#C0504D"); a1.set_xlabel("Epoch")
a1.set_ylabel("Weighted BCE loss"); a1.set_title("Training loss"); a1.grid(alpha=.3)
a2.plot(ep, auc, marker="o", color="#3A6EA5"); a2.set_xlabel("Epoch")
a2.set_ylabel("Mean AUC"); a2.set_title("Validation mean AUC"); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
mean_auc, aucs, P, Y = evaluate(model, val_loader)
s = pd.Series(aucs).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(s)), s.values, color="#3A6EA5")
ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index, fontsize=9)
ax.axvline(0.5, color="gray", ls="--", lw=1)
ax.axvline(mean_auc, color="#C0504D", ls="--", lw=1.2, label=f"mean {mean_auc:.3f}")
ax.set_xlim(0.4, 1.0); ax.set_xlabel("AUC"); ax.set_title("Per-class AUC (validation)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for i, c in enumerate(LABEL_COLS):
    if c not in aucs:
        continue
    fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
    ax.plot(fpr, tpr, lw=1.2, label=f"{c} ({aucs[c]:.2f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves (validation)")
ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()

## 7. Test set

In [ ]:
if len(test_df) > 0:
    test_loader = DataLoader(CXRDataset(test_df, eval_tf), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    ck = torch.load(os.path.join(CKPT_DIR, "best.pt"), map_location=device)
    model.load_state_dict(ck["model"])
    t_auc, t_aucs, _, _ = evaluate(model, test_loader)
    print(f"TEST mean AUC: {t_auc:.4f}  (n={len(test_df)})")
    print(pd.Series(t_aucs).sort_values(ascending=False).round(4).to_string())
else:
    print("No test images downloaded yet.")